In [1]:
import os
import math
import time
from typing import Any, Dict, List, Tuple

import numpy as np

# Configure XLA client allocation behavior
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax
import jax.numpy as jnp
from jax import random

# Compatibility patch for JAX core tracing extensions
try:
    import jax.extend.core as jec
    if not hasattr(jax.core, "get_opaque_trace_state") and hasattr(jec, "get_opaque_trace_state"):
        jax.core.get_opaque_trace_state = jec.get_opaque_trace_state
except Exception:
    pass

import flax.linen as nn
from flax.training import train_state
import optax

# -----------------------------------------------------------------------------
# 0. Global Setup & Structural Constants
# -----------------------------------------------------------------------------
DEVICES = jax.devices()
SEEDS: List[int] = [42, 1337, 2026]
DTYPE: jnp.dtype = jnp.bfloat16

D_MODEL: int = 128
D_INTER: int = 384
M_MODES: int = 4
N_LAYERS: int = 2
BATCH_SIZE: int = 512
LEARNING_RATE: float = 1e-3

# -----------------------------------------------------------------------------
# 1. Hyperspherical & Lie-Algebraic Primitives
# -----------------------------------------------------------------------------
def to_unit_sphere(x: jnp.ndarray, eps: float = 1e-7) -> jnp.ndarray:
    """
    Retracts Euclidean vectors onto the unit hypersphere S^{d-1}:
        Pi_S(x) = x / (||x||_2 + eps)
    """
    norm = jnp.sqrt(jnp.sum(x.astype(jnp.float32)**2, axis=-1, keepdims=True) + eps**2)
    return (x.astype(jnp.float32) / norm).astype(x.dtype)


def lie_bivector_action(u: jnp.ndarray, v: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """
    Computes the rank-2 antisymmetric commutator action (u ^ v)x in so(d):
        (u v^T - v u^T) x = (v^T x) u - (u^T x) v
    Ensures exact orthogonality: <x, (u ^ v)x> = 0.
    """
    u_f, v_f, x_f = u.astype(jnp.float32), v.astype(jnp.float32), x.astype(jnp.float32)
    vx = jnp.sum(v_f * x_f, axis=-1, keepdims=True)
    ux = jnp.sum(u_f * x_f, axis=-1, keepdims=True)
    tangent_vector = vx * u_f - ux * v_f
    return tangent_vector.astype(x.dtype)


def unit_riemannian_expmap(
    x: jnp.ndarray,
    v_tangent: jnp.ndarray,
    max_angle: float = math.pi / 2.0,
    eps: float = 1e-6
) -> jnp.ndarray:
    """
    Closed-form Riemannian exponential map on S^{d-1}:
        Exp_x(v) = cos(||v||) x + sinc(||v||) v
    Regularized to prevent 0/0 indeterminate derivatives at the origin.
    """
    x_f, v_f = x.astype(jnp.float32), v_tangent.astype(jnp.float32)
    v_norm = jnp.sqrt(jnp.sum(v_f**2, axis=-1, keepdims=True) + eps**2)
    v_norm_clamped = jnp.clip(v_norm, 0.0, max_angle)

    sinc = jnp.where(v_norm > eps, jnp.sin(v_norm_clamped) / v_norm, 1.0)
    x_next = jnp.cos(v_norm_clamped) * x_f + sinc * v_f

    norm_next = jnp.sqrt(jnp.sum(x_next**2, axis=-1, keepdims=True) + eps**2)
    return (x_next / norm_next).astype(x.dtype)


class PureLieVolterraFlowLayer(nn.Module):
    """
    Causal Lie-Volterra layer combining Gauss-Chebyshev spectral memory,
    quadratic functional Volterra cross-interactions, and Lie algebra so(d) updates.
    """
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    m_modes: int = M_MODES
    block_size: int = 3
    dtype: jnp.dtype = DTYPE

    def setup(self) -> None:
        t = jnp.arange(self.block_size, dtype=jnp.float32)
        m = jnp.arange(self.m_modes, dtype=jnp.float32)[None, :]
        t_norm = (2.0 * t[:, None] / max(self.block_size - 1, 1)) - 1.0
        t_clipped = jnp.clip(t_norm, -1.0 + 1e-6, 1.0 - 1e-6)
        cheb = jnp.cos(m * jnp.arccos(t_clipped))

        x_s = (2.0 * t + 1.0) / (2.0 * self.block_size)
        w_cheb = 1.0 / jnp.sqrt(x_s * (1.0 - x_s) + 1e-5)
        w_cheb = (w_cheb / jnp.mean(w_cheb))[:, None]

        self.base_phi = cheb
        self.base_psi = cheb * w_cheb

        t_grid = t[:, None]
        s_grid = t[None, :]
        self.delta_ts = jnp.maximum(t_grid - s_grid, 0.0)
        self.causal_mask = jnp.tril(jnp.ones((self.block_size, self.block_size), dtype=jnp.bool_))

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        b, t, d = x.shape

        w_curr = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="w_curr")
        j_current = w_curr(x)

        init_gammas = jnp.linspace(math.log(0.005), math.log(0.20), self.m_modes)
        log_gamma = self.param("log_gamma", lambda rng: init_gammas)
        lambdas = self.param("lambdas", lambda rng: jnp.ones((self.m_modes,), dtype=jnp.float32) / math.sqrt(self.m_modes))

        phi = self.base_phi[:t, :]
        psi = self.base_psi[:t, :]

        gamma = jnp.clip(jnp.exp(log_gamma), 1e-4, 2.0)
        decay = jnp.exp(-gamma[:, None, None] * self.delta_ts[:t, :t][None, :, :])
        base_kernel = jnp.einsum("tm,sm->mst", phi, psi)
        causal_kernel = jnp.sum(lambdas[:, None, None] * base_kernel * decay, axis=0)
        causal_kernel = jnp.where(self.causal_mask[:t, :t], causal_kernel, 0.0)

        h_memory = jnp.matmul(causal_kernel.astype(self.dtype)[None, :, :], j_current)

        w_u = nn.Dense(self.d_inter, use_bias=False, dtype=self.dtype, name="w_u")
        w_v = nn.Dense(self.d_inter, use_bias=False, dtype=self.dtype, name="w_v")
        w_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="w_proj")

        u_field = nn.silu(w_u(x))
        v_field = nn.silu(w_v(h_memory))

        interaction = u_field * v_field
        force_direction = w_proj(interaction)

        v_tangent = lie_bivector_action(force_direction, force_direction * 0.5 + x, x)

        eta = self.param("tangent_scale", lambda rng: jnp.array(0.05, dtype=jnp.float32))
        return unit_riemannian_expmap(x, v_tangent * eta)


def build_decay_mask(params: Dict[str, Any]) -> Any:
    """
    Strict parameter masking: disables weight decay for scalar and metric parameters
    (log_tau, log_gamma, lambdas, tangent_scale, bias) to prevent manifold collapse.
    """
    def should_decay(path, param):
        path_str = "/".join([str(p.key) if hasattr(p, 'key') else str(p) for p in path])
        if any(non_decay in path_str for non_decay in ["log_tau", "log_gamma", "lambdas", "tangent_scale", "bias"]):
            return False
        return True

    return jax.tree_util.tree_map_with_path(should_decay, params)


print(f"[SETUP] Hardware: {DEVICES[0].device_kind.upper()} | Backend: {jax.default_backend()} | Precision: bfloat16")

[SETUP] Hardware: NVIDIA L4 | Backend: gpu | Precision: bfloat16


In [5]:
# -----------------------------------------------------------------------------
# TASK 1: MODULAR ADDITION (a + b mod 97)
# -----------------------------------------------------------------------------
P_ADD: int = 97
MAX_STEPS_ADD: int = 10000
LOG_INTERVAL_ADD: int = 250
EARLY_STOP_VAL_ACC_ADD: float = 0.9999
WEIGHT_DECAY_ADD: float = 0.1


class PLVFModularAdditionModel(nn.Module):
    vocab_size: int = P_ADD + 1
    target_classes: int = P_ADD
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = 3
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, idx: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), self.dtype)
        tok_u = to_unit_sphere(tok_embed)
        x = tok_u[idx]

        for i in range(self.n_layers):
            x = PureLieVolterraFlowLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x)

        last_state = to_unit_sphere(x[:, -1, :])
        target_vocab_u = tok_u[:self.target_classes, :]
        cos_sim = jnp.matmul(last_state.astype(jnp.float32), target_vocab_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(15.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 1.0, 40.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_modular_addition_data(seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    pairs = [(a, b) for a in range(P_ADD) for b in range(P_ADD)]
    rng.shuffle(pairs)

    n_train = len(pairs) // 2
    train_pairs, val_pairs = pairs[:n_train], pairs[n_train:]

    def build(pair_list):
        x = np.zeros((len(pair_list), 3), dtype=np.int32)
        y = np.zeros((len(pair_list),), dtype=np.int32)
        for i, (a, b) in enumerate(pair_list):
            x[i, 0], x[i, 1], x[i, 2] = a, b, P_ADD
            y[i] = (a + b) % P_ADD
        return jnp.array(x), jnp.array(y)

    x_train, y_train = build(train_pairs)
    x_val, y_val = build(val_pairs)
    return x_train, y_train, x_val, y_val


def run_modular_addition_benchmark():
    print("=" * 125)
    print(f"BENCHMARK: MODULAR ADDITION (Z_{P_ADD}) | 50% TRAIN SPLIT | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_ADD}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_ADD}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, solve_steps, thetas, times = [], [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_modular_addition_data(seed=seed)
        model = PLVFModularAdditionModel()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, 3), dtype=jnp.int32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        decay_mask = build_decay_mask(params)
        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_ADD, b1=0.9, b2=0.98, mask=decay_mask)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        # XLA Graph Warmup
        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        solve_step = None
        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_ADD + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_ADD == 0 or step == MAX_STEPS_ADD:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

                if v_acc >= EARLY_STOP_VAL_ACC_ADD:
                    solve_step = step
                    break

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        solve_steps.append(solve_step if solve_step else MAX_STEPS_ADD)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            step_disp = f"Step {solve_step}" if solve_step else f"> {MAX_STEPS_ADD}"
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | {step_disp:<12} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: MODULAR ADDITION (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"{int(np.mean(solve_steps))}" if all(s < MAX_STEPS_ADD for s in solve_steps) else f"> {MAX_STEPS_ADD}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Modular Addition (Z_97)':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_modular_addition_benchmark()

BENCHMARK: MODULAR ADDITION (Z_97) | 50% TRAIN SPLIT | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=0.1) | Batch: 512 | Max Steps: 10000
Parameters: 340.2k | Train Samples: 4704 | Val Samples: 4705
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   250   |     4.4565   |      2.14%   |     4.9568   |      0.20%   |  91.08 deg  |  12.3 |    3.066   |   8.5s
   500   |     4.2132   |      2.32%   |     5.8317   |      0.00%   |  94.27 deg  |  11.0 |    1.972   |   9.8s
   750   |     3.9708   |      6.12%   |     6.4888   |      0.00%   |  95.36 deg  |  12.4 |    1.837   |  11.0s
  1000   |     1.1323   |     79.54%   |     4.3099   |     13.60%   |  81.23 deg  |  17.0 |    2.946   |  

In [6]:
# -----------------------------------------------------------------------------
# TASK 2: MODULAR MULTIPLICATION (a * b mod 97)
# -----------------------------------------------------------------------------
P_MUL: int = 97
MAX_STEPS_MUL: int = 10000
LOG_INTERVAL_MUL: int = 250
EARLY_STOP_VAL_ACC_MUL: float = 0.9999
WEIGHT_DECAY_MUL: float = 0.1


class PLVFModularMultiplicationModel(nn.Module):
    vocab_size: int = P_MUL + 1
    target_classes: int = P_MUL
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = 3
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, idx: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), self.dtype)
        tok_u = to_unit_sphere(tok_embed)
        x = tok_u[idx]

        for i in range(self.n_layers):
            x = PureLieVolterraFlowLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x)

        last_state = to_unit_sphere(x[:, -1, :])
        target_vocab_u = tok_u[:self.target_classes, :]
        cos_sim = jnp.matmul(last_state.astype(jnp.float32), target_vocab_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(15.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 1.0, 40.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_modular_multiplication_data(seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    pairs = [(a, b) for a in range(P_MUL) for b in range(P_MUL)]
    rng.shuffle(pairs)

    n_train = len(pairs) // 2
    train_pairs, val_pairs = pairs[:n_train], pairs[n_train:]

    def build(pair_list):
        x = np.zeros((len(pair_list), 3), dtype=np.int32)
        y = np.zeros((len(pair_list),), dtype=np.int32)
        for i, (a, b) in enumerate(pair_list):
            x[i, 0], x[i, 1], x[i, 2] = a, b, P_MUL
            y[i] = (a * b) % P_MUL
        return jnp.array(x), jnp.array(y)

    x_train, y_train = build(train_pairs)
    x_val, y_val = build(val_pairs)
    return x_train, y_train, x_val, y_val


def run_modular_multiplication_benchmark():
    print("=" * 125)
    print(f"BENCHMARK: MODULAR MULTIPLICATION (Z_{P_MUL}) | 50% TRAIN SPLIT | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_MUL}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_MUL}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, solve_steps, thetas, times = [], [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_modular_multiplication_data(seed=seed)
        model = PLVFModularMultiplicationModel()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, 3), dtype=jnp.int32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        decay_mask = build_decay_mask(params)
        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_MUL, b1=0.9, b2=0.98, mask=decay_mask)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        # XLA Graph Warmup
        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        solve_step = None
        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_MUL + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_MUL == 0 or step == MAX_STEPS_MUL:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

                if v_acc >= EARLY_STOP_VAL_ACC_MUL:
                    solve_step = step
                    break

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        solve_steps.append(solve_step if solve_step else MAX_STEPS_MUL)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            step_disp = f"Step {solve_step}" if solve_step else f"> {MAX_STEPS_MUL}"
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | {step_disp:<12} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: MODULAR MULTIPLICATION (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"{int(np.mean(solve_steps))}" if all(s < MAX_STEPS_MUL for s in solve_steps) else f"> {MAX_STEPS_MUL}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Modular Mult (Z_97)':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_modular_multiplication_benchmark()

BENCHMARK: MODULAR MULTIPLICATION (Z_97) | 50% TRAIN SPLIT | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=0.1) | Batch: 512 | Max Steps: 10000
Parameters: 340.2k | Train Samples: 4704 | Val Samples: 4705
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   250   |     4.4543   |      3.72%   |     4.5880   |      2.95%   |  88.91 deg  |  12.3 |    3.930   |   8.5s
   500   |     3.0670   |     15.62%   |     3.8571   |      3.99%   |  76.73 deg  |  13.2 |    7.679   |   9.8s
   750   |     1.8697   |     33.70%   |     3.0376   |      5.83%   |  55.66 deg  |  17.4 |    2.744   |  11.1s
  1000   |     0.6152   |     83.00%   |     2.4586   |     25.70%   |  61.15 deg  |  22.4 |    2.050

In [10]:
# -----------------------------------------------------------------------------
# TASK 3: PERMUTATIONS S_5 (Non-Abelian Group Composition, 120 Elements)
# -----------------------------------------------------------------------------
import itertools

N_PERMS: int = 120
MAX_STEPS_S5: int = 15000
LOG_INTERVAL_S5: int = 500
EARLY_STOP_VAL_ACC_S5: float = 0.9999
WEIGHT_DECAY_S5: float = 1.0


class PLVFS5Model(nn.Module):
    vocab_size: int = N_PERMS + 1
    target_classes: int = N_PERMS
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = 3
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, idx: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), self.dtype)
        tok_u = to_unit_sphere(tok_embed)
        x = tok_u[idx]

        for i in range(self.n_layers):
            x = PureLieVolterraFlowLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x)

        last_state = to_unit_sphere(x[:, -1, :])
        target_vocab_u = tok_u[:self.target_classes, :]
        cos_sim = jnp.matmul(last_state.astype(jnp.float32), target_vocab_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(18.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 2.5, 30.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_s5_permutation_data(seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    perms = list(itertools.permutations(range(5)))
    perm_to_idx = {p: i for i, p in enumerate(perms)}

    pairs = [(i, j) for i in range(N_PERMS) for j in range(N_PERMS)]
    rng.shuffle(pairs)

    n_train = len(pairs) // 2
    train_pairs, val_pairs = pairs[:n_train], pairs[n_train:]

    def build(pair_list):
        x = np.zeros((len(pair_list), 3), dtype=np.int32)
        y = np.zeros((len(pair_list),), dtype=np.int32)
        for idx, (i, j) in enumerate(pair_list):
            p1, p2 = perms[i], perms[j]
            comp = tuple(p1[p2[k]] for k in range(5))
            x[idx, 0], x[idx, 1], x[idx, 2] = i, j, N_PERMS
            y[idx] = perm_to_idx[comp]
        return jnp.array(x), jnp.array(y)

    x_train, y_train = build(train_pairs)
    x_val, y_val = build(val_pairs)
    return x_train, y_train, x_val, y_val


def run_s5_benchmark():
    print("=" * 125)
    print("BENCHMARK: PERMUTATIONS S_5 (NON-ABELIAN GROUP) | 50% TRAIN SPLIT | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_S5}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_S5}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, solve_steps, thetas, times = [], [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_s5_permutation_data(seed=seed)
        model = PLVFS5Model()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, 3), dtype=jnp.int32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_S5, b1=0.9, b2=0.98)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        solve_step = None
        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_S5 + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_S5 == 0 or step == MAX_STEPS_S5:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

                if v_acc >= EARLY_STOP_VAL_ACC_S5:
                    solve_step = step
                    break

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        solve_steps.append(solve_step if solve_step else MAX_STEPS_S5)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            step_disp = f"Step {solve_step}" if solve_step else f"> {MAX_STEPS_S5}"
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | {step_disp:<12} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: PERMUTATIONS S_5 (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"{int(np.mean(solve_steps))}" if all(s < MAX_STEPS_S5 for s in solve_steps) else f"> {MAX_STEPS_S5}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Permutations S_5':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_s5_benchmark()

BENCHMARK: PERMUTATIONS S_5 (NON-ABELIAN GROUP) | 50% TRAIN SPLIT | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=1.0) | Batch: 512 | Max Steps: 15000
Parameters: 343.2k | Train Samples: 7200 | Val Samples: 7200
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   500   |     4.6454   |      1.71%   |     5.1794   |      0.01%   |  93.05 deg  |   4.9 |    1.044   |   8.3s
  1000   |     4.5574   |      1.80%   |     5.3447   |      0.00%   |  95.94 deg  |   3.8 |    1.198   |  10.8s
  1500   |     4.5006   |      3.28%   |     5.3301   |      0.00%   |  96.87 deg  |   3.3 |    1.551   |  13.2s
  2000   |     4.4951   |      4.06%   |     5.2911   |      0.03%   |  97.04 deg  |   3.1 |  

In [9]:
# -----------------------------------------------------------------------------
# TASK 4: FORMAL LANGUAGE DYCK-1 (Balanced Parentheses Stack, Sequence Length 12)
# -----------------------------------------------------------------------------
SEQ_LEN_DYCK: int = 12
VOCAB_SIZE_DYCK: int = 4  # 0: '(', 1: ')', 2: '=' (EOS), 3: PAD
BLOCK_SIZE_DYCK: int = SEQ_LEN_DYCK + 1
MAX_STEPS_DYCK: int = 10000
LOG_INTERVAL_DYCK: int = 250
EARLY_STOP_VAL_ACC_DYCK: float = 0.9999
WEIGHT_DECAY_DYCK: float = 0.05


class PLVFDyckModel(nn.Module):
    vocab_size: int = VOCAB_SIZE_DYCK
    n_classes: int = 2
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = BLOCK_SIZE_DYCK
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, idx: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), self.dtype)
        tok_u = to_unit_sphere(tok_embed)
        x = tok_u[idx]

        for i in range(self.n_layers):
            x = PureLieVolterraFlowLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x)

        last_state = to_unit_sphere(x[:, -1, :])
        w_classes = self.param("w_classes", nn.initializers.normal(stddev=0.02), (self.n_classes, self.d_model))
        w_classes_u = to_unit_sphere(w_classes)
        cos_sim = jnp.matmul(last_state.astype(jnp.float32), w_classes_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(15.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 1.0, 40.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_dyck1_data(n_samples: int = 8000, seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    half = n_samples // 2
    samples, labels = [], []

    def generate_balanced(length):
        res = []
        depth = 0
        for step in range(length):
            rem = length - step
            if depth == 0:
                res.append(0)
                depth += 1
            elif depth == rem:
                res.append(1)
                depth -= 1
            else:
                choice = rng.choice([0, 1])
                res.append(choice)
                depth += 1 if choice == 0 else -1
        return res

    for _ in range(half):
        samples.append(generate_balanced(SEQ_LEN_DYCK))
        labels.append(1)

    for _ in range(half):
        cand = rng.integers(0, 2, size=SEQ_LEN_DYCK).tolist()
        d = 0
        is_bal = True
        for b in cand:
            d += 1 if b == 0 else -1
            if d < 0:
                is_bal = False
                break
        if d != 0:
            is_bal = False
        samples.append(cand)
        labels.append(1 if is_bal else 0)

    indices = np.arange(n_samples)
    rng.shuffle(indices)

    x_all = np.zeros((n_samples, BLOCK_SIZE_DYCK), dtype=np.int32)
    y_all = np.array([labels[i] for i in indices], dtype=np.int32)

    for idx, sample_idx in enumerate(indices):
        x_all[idx, :SEQ_LEN_DYCK] = samples[sample_idx]
        x_all[idx, SEQ_LEN_DYCK] = 2  # EOS token

    split = int(0.7 * n_samples)
    x_train, y_train = jnp.array(x_all[:split]), jnp.array(y_all[:split])
    x_val, y_val = jnp.array(x_all[split:]), jnp.array(y_all[split:])
    return x_train, y_train, x_val, y_val


def run_dyck_benchmark():
    print("=" * 125)
    print("BENCHMARK: FORMAL LANGUAGE DYCK-1 (STACK DEPTH) | 70/30 SPLIT | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_DYCK}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_DYCK}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, solve_steps, thetas, times = [], [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_dyck1_data(seed=seed)
        model = PLVFDyckModel()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, BLOCK_SIZE_DYCK), dtype=jnp.int32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        decay_mask = build_decay_mask(params)
        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_DYCK, b1=0.9, b2=0.98, mask=decay_mask)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        # XLA Graph Warmup
        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        solve_step = None
        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_DYCK + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_DYCK == 0 or step == MAX_STEPS_DYCK:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

                if v_acc >= EARLY_STOP_VAL_ACC_DYCK:
                    solve_step = step
                    break

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        solve_steps.append(solve_step if solve_step else MAX_STEPS_DYCK)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            step_disp = f"Step {solve_step}" if solve_step else f"> {MAX_STEPS_DYCK}"
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | {step_disp:<12} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: FORMAL LANGUAGE DYCK-1 (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"{int(np.mean(solve_steps))}" if all(s < MAX_STEPS_DYCK for s in solve_steps) else f"> {MAX_STEPS_DYCK}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Dyck-1 Parentheses':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_dyck_benchmark()

BENCHMARK: FORMAL LANGUAGE DYCK-1 (STACK DEPTH) | 70/30 SPLIT | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=0.05) | Batch: 512 | Max Steps: 10000
Parameters: 328.5k | Train Samples: 5600 | Val Samples: 2400
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   250   |     0.4652   |     82.15%   |     0.4665   |     82.46%   |  69.50 deg  |  15.7 |   85.908   |  37.0s
   500   |     0.0162   |     99.98%   |     0.0156   |    100.00%   |  62.88 deg  |  15.9 |    1.141   |  39.1s
  Seed 1337: Val Acc = 100.00% | Theta Err = 54.53 deg | Step 500     |  14.6s
  Seed 2026: Val Acc = 100.00% | Theta Err = 44.76 deg | Step 750     |  15.6s

EMPIRICAL SUMMARY: FORMAL LANGUAGE DYCK-1 (3 SEEDS)

In [11]:
# -----------------------------------------------------------------------------
# TASK 5: SPARSE PARITY (k=3 bits in N=20, Barak et al. Dense Bipolar Regime)
# -----------------------------------------------------------------------------
N_BITS_PARITY: int = 20
K_RELEVANT_PARITY: int = 3
TARGET_INDICES_PARITY: List[int] = [2, 7, 11]

N_TRAIN_PARITY: int = 12000
N_VAL_PARITY: int = 4000
MAX_STEPS_PARITY: int = 10000
LOG_INTERVAL_PARITY: int = 250
EARLY_STOP_VAL_ACC_PARITY: float = 0.9999
WEIGHT_DECAY_PARITY: float = 0.01


class LieSphericalParityLayer(nn.Module):
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        w_u = nn.Dense(self.d_inter, use_bias=False, dtype=self.dtype, name="w_u")
        w_v = nn.Dense(self.d_inter, use_bias=False, dtype=self.dtype, name="w_v")
        w_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="w_proj")

        u = nn.silu(w_u(x))
        v = nn.silu(w_v(x))
        force = w_proj(u * v)

        v_tangent = lie_bivector_action(force, force * 0.5 + x, x)
        eta = self.param("tangent_scale", lambda rng: jnp.array(0.10, dtype=jnp.float32))
        return unit_riemannian_expmap(x, v_tangent * eta)


class PLVFSparseParityModel(nn.Module):
    n_bits: int = N_BITS_PARITY
    n_classes: int = 2
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        w_in = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="w_in")
        x_sphere = to_unit_sphere(w_in(x))

        for i in range(self.n_layers):
            x_sphere = LieSphericalParityLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x_sphere)

        w_classes = self.param("w_classes", nn.initializers.normal(stddev=0.02), (self.n_classes, self.d_model), self.dtype)
        w_classes_u = to_unit_sphere(w_classes)
        cos_sim = jnp.matmul(x_sphere.astype(jnp.float32), w_classes_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(15.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 1.0, 40.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_dense_parity_data(seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    n_total = N_TRAIN_PARITY + N_VAL_PARITY
    raw_bits = rng.choice([-1.0, 1.0], size=(n_total, N_BITS_PARITY)).astype(np.float32)
    parity_prod = np.prod(raw_bits[:, TARGET_INDICES_PARITY], axis=-1)
    labels = np.where(parity_prod > 0, 1, 0).astype(np.int32)

    x_train, y_train = jnp.array(raw_bits[:N_TRAIN_PARITY]), jnp.array(labels[:N_TRAIN_PARITY])
    x_val, y_val = jnp.array(raw_bits[N_TRAIN_PARITY:]), jnp.array(labels[N_TRAIN_PARITY:])
    return x_train, y_train, x_val, y_val


def run_sparse_parity_benchmark():
    print("=" * 125)
    print(f"BENCHMARK: SPARSE PARITY (k={K_RELEVANT_PARITY}, N={N_BITS_PARITY}) | DENSE BIPOLAR | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_PARITY}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_PARITY}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, solve_steps, thetas, times = [], [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_dense_parity_data(seed=seed)
        model = PLVFSparseParityModel()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, N_BITS_PARITY), dtype=jnp.float32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        decay_mask = build_decay_mask(params)
        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_PARITY, b1=0.9, b2=0.98, mask=decay_mask)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        # XLA Graph Warmup
        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        solve_step = None
        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_PARITY + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_PARITY == 0 or step == MAX_STEPS_PARITY:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

                if v_acc >= EARLY_STOP_VAL_ACC_PARITY:
                    solve_step = step
                    break

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        solve_steps.append(solve_step if solve_step else MAX_STEPS_PARITY)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            step_disp = f"Step {solve_step}" if solve_step else f"> {MAX_STEPS_PARITY}"
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | {step_disp:<12} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: SPARSE PARITY (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"{int(np.mean(solve_steps))}" if all(s < MAX_STEPS_PARITY for s in solve_steps) else f"> {MAX_STEPS_PARITY}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Sparse Parity (k=3, N=20)':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_sparse_parity_benchmark()

BENCHMARK: SPARSE PARITY (k=3, N=20) | DENSE BIPOLAR | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=0.01) | Batch: 512 | Max Steps: 10000
Parameters: 297.7k | Train Samples: 12000 | Val Samples: 4000
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   250   |     0.0001   |    100.00%   |     0.0001   |    100.00%   |  69.33 deg  |  15.5 |    0.002   |  22.2s
  Seed 1337: Val Acc = 100.00% | Theta Err = 70.81 deg | Step 250     |   7.8s
  Seed 2026: Val Acc = 100.00% | Theta Err = 69.16 deg | Step 250     |   8.0s

EMPIRICAL SUMMARY: SPARSE PARITY (3 SEEDS)
TASK                       | PARAMS     | VAL ACC (MEAN +- STD)  | THETA ERR (MEAN)   | SOLVE STEP (MEAN)  | TIME
---------------

In [12]:
# -----------------------------------------------------------------------------
# TASK 6: RANDOM LOOKUP HASH (Negative Control / Pure Tabular Memorization)
# -----------------------------------------------------------------------------
P_HASH: int = 64
MAX_STEPS_HASH: int = 2500
LOG_INTERVAL_HASH: int = 250
WEIGHT_DECAY_HASH: float = 0.0  # Zero weight decay to evaluate unconstrained memorization


class PLVFRandomLookupModel(nn.Module):
    vocab_size: int = P_HASH + 1
    target_classes: int = P_HASH
    d_model: int = D_MODEL
    d_inter: int = D_INTER
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    block_size: int = 3
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, idx: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), self.dtype)
        tok_u = to_unit_sphere(tok_embed)
        x = tok_u[idx]

        for i in range(self.n_layers):
            x = PureLieVolterraFlowLayer(
                d_model=self.d_model,
                d_inter=self.d_inter,
                m_modes=self.m_modes,
                block_size=self.block_size,
                dtype=self.dtype,
                name=f"flow_{i}"
            )(x)

        last_state = to_unit_sphere(x[:, -1, :])
        target_vocab_u = tok_u[:self.target_classes, :]
        cos_sim = jnp.matmul(last_state.astype(jnp.float32), target_vocab_u.astype(jnp.float32).T)

        log_tau = self.param("log_tau", lambda rng: jnp.array(math.log(18.0), dtype=jnp.float32))
        tau = jnp.clip(jnp.exp(log_tau), 1.0, 40.0)
        logits = tau * cos_sim
        return logits, cos_sim


def get_random_lookup_data(seed: int = 42) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rng = np.random.default_rng(seed)
    random_table = rng.integers(0, P_HASH, size=(P_HASH, P_HASH))

    pairs = [(a, b) for a in range(P_HASH) for b in range(P_HASH)]
    rng.shuffle(pairs)

    n_train = len(pairs) // 2
    train_pairs, val_pairs = pairs[:n_train], pairs[n_train:]

    def build(pair_list):
        x = np.zeros((len(pair_list), 3), dtype=np.int32)
        y = np.zeros((len(pair_list),), dtype=np.int32)
        for i, (a, b) in enumerate(pair_list):
            x[i, 0], x[i, 1], x[i, 2] = a, b, P_HASH
            y[i] = random_table[a, b]
        return jnp.array(x), jnp.array(y)

    x_train, y_train = build(train_pairs)
    x_val, y_val = build(val_pairs)
    return x_train, y_train, x_val, y_val


def run_random_hash_benchmark():
    print("=" * 125)
    print(f"BENCHMARK: RANDOM LOOKUP HASH (P={P_HASH}) | NEGATIVE CONTROL | 3 SEEDS")
    print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY_HASH}) | Batch: {BATCH_SIZE} | Max Steps: {MAX_STEPS_HASH}")
    print("=" * 125)

    @jax.jit
    def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        def loss_fn(params):
            logits, cos_sim = state.apply_fn({"params": params}, x)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
            return loss, (logits, cos_sim)

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, (logits, cos_sim)), grads = grad_fn(state.params)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        gnorm = optax.global_norm(grads)
        return state.apply_gradients(grads=grads), loss, acc, gnorm

    @jax.jit
    def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        logits, cos_sim = state.apply_fn({"params": state.params}, x)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y).mean()
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)

        num_classes = cos_sim.shape[-1]
        one_hot = jax.nn.one_hot(y, num_classes)
        s_y = jnp.sum(cos_sim * one_hot, axis=-1)
        theta_rad = jnp.arccos(jnp.clip(s_y, -1.0 + 1e-6, 1.0 - 1e-6))
        theta_deg = jnp.mean(theta_rad * (180.0 / math.pi))
        return loss, acc, theta_deg

    def evaluate_full(state: train_state.TrainState, x_data: jnp.ndarray, y_data: jnp.ndarray) -> Tuple[float, float, float]:
        n = len(x_data)
        losses, accs, thetas = [], [], []
        for i in range(0, n, BATCH_SIZE):
            bx = x_data[i : i + BATCH_SIZE]
            by = y_data[i : i + BATCH_SIZE]
            l, a, th = eval_step(state, bx, by)
            losses.append(float(l))
            accs.append(float(a))
            thetas.append(float(th))
        return float(np.mean(losses)), float(np.mean(accs)), float(np.mean(thetas))

    val_accs, thetas, times = [], [], []
    param_count = 0

    for seed in SEEDS:
        t0 = time.time()
        x_train, y_train, x_val, y_val = get_random_lookup_data(seed=seed)
        model = PLVFRandomLookupModel()

        init_rng = random.PRNGKey(seed)
        dummy_x = jnp.ones((BATCH_SIZE, 3), dtype=jnp.int32)
        params = model.init(init_rng, dummy_x)["params"]
        param_count = sum(p.size for p in jax.tree_util.tree_leaves(params))

        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY_HASH, b1=0.9, b2=0.98)
        )
        state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

        # XLA Graph Warmup
        train_rng = random.PRNGKey(seed)
        train_rng, sub_rng = random.split(train_rng)
        dummy_idx = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=len(x_train))
        state, _, _, _ = train_step(state, x_train[dummy_idx], y_train[dummy_idx])
        _ = eval_step(state, x_train[dummy_idx], y_train[dummy_idx])
        jax.block_until_ready(state.params)

        if seed == SEEDS[0]:
            print(f"Parameters: {param_count/1e3:.1f}k | Train Samples: {len(x_train)} | Val Samples: {len(x_val)}")
            print("-" * 125)
            print(f"{'STEP':<8} | {'TRAIN LOSS':<12} | {'TRAIN ACC':<12} | {'VAL LOSS':<12} | {'VAL ACC':<12} | {'THETA ERR':<10} | {'TAU':<6} | {'GRAD NORM':<10} | {'TIME'}")
            print("-" * 125)

        last_v_acc, last_v_theta = 0.0, 180.0
        n_train = len(x_train)

        for step in range(1, MAX_STEPS_HASH + 1):
            train_rng, sub_rng = random.split(train_rng)
            indices = random.randint(sub_rng, shape=(BATCH_SIZE,), minval=0, maxval=n_train)
            state, loss_val, acc_val, gnorm = train_step(state, x_train[indices], y_train[indices])

            if step % LOG_INTERVAL_HASH == 0 or step == MAX_STEPS_HASH:
                jax.block_until_ready(state.params)
                t_loss, t_acc, _ = evaluate_full(state, x_train, y_train)
                v_loss, v_acc, v_theta = evaluate_full(state, x_val, y_val)
                tau_val = float(np.exp(state.params["log_tau"]))
                elapsed = time.time() - t0

                last_v_acc, last_v_theta = v_acc, v_theta

                if seed == SEEDS[0]:
                    print(
                        f"{step:>6}   | {t_loss:>10.4f}   | {t_acc*100:>9.2f}%   | "
                        f"{v_loss:>10.4f}   | {v_acc*100:>9.2f}%   | {v_theta:>6.2f} deg  | "
                        f"{tau_val:>5.1f} | {float(gnorm):>8.3f}   | {elapsed:>5.1f}s"
                    )

        elapsed = time.time() - t0
        val_accs.append(last_v_acc * 100.0)
        thetas.append(last_v_theta)
        times.append(elapsed)

        if seed != SEEDS[0]:
            print(f"  Seed {seed:>4}: Val Acc = {last_v_acc*100:>6.2f}% | Theta Err = {last_v_theta:>5.2f} deg | Step > {MAX_STEPS_HASH:<5} | {elapsed:>5.1f}s")

    print("\n" + "=" * 125)
    print("EMPIRICAL SUMMARY: RANDOM LOOKUP HASH (3 SEEDS)")
    print("=" * 125)
    print(f"{'TASK':<26} | {'PARAMS':<10} | {'VAL ACC (MEAN +- STD)':<22} | {'THETA ERR (MEAN)':<18} | {'SOLVE STEP (MEAN)':<18} | {'TIME'}")
    print("-" * 125)
    acc_str = f"{np.mean(val_accs):>5.2f}% +- {np.std(val_accs):>4.2f}%"
    th_str = f"{np.mean(thetas):>5.2f} deg"
    step_str = f"> {MAX_STEPS_HASH}"
    time_str = f"{np.mean(times):>5.1f}s"
    print(f"{'Random Lookup Hash':<26} | {param_count/1e3:>5.1f}k     | {acc_str:<22} | {th_str:<18} | {step_str:<18} | {time_str}")
    print("=" * 125)


if __name__ == "__main__":
    run_random_hash_benchmark()

BENCHMARK: RANDOM LOOKUP HASH (P=64) | NEGATIVE CONTROL | 3 SEEDS
Optimizer: AdamW(lr=0.001, wd=0.0) | Batch: 512 | Max Steps: 2500
Parameters: 336.0k | Train Samples: 2048 | Val Samples: 2048
-----------------------------------------------------------------------------------------------------------------------------
STEP     | TRAIN LOSS   | TRAIN ACC    | VAL LOSS     | VAL ACC      | THETA ERR  | TAU    | GRAD NORM  | TIME
-----------------------------------------------------------------------------------------------------------------------------
   250   |     3.5714   |      6.93%   |     4.9618   |      1.51%   |  89.06 deg  |  16.1 |    6.713   |   8.9s
   500   |     3.1119   |     12.94%   |     5.7927   |      1.51%   |  89.96 deg  |  16.1 |    3.445   |  10.1s
   750   |     0.2178   |     99.56%   |     7.9623   |      1.86%   |  89.83 deg  |  22.0 |    2.780   |  11.3s
  1000   |     0.0020   |    100.00%   |     9.8084   |      1.37%   |  89.84 deg  |  25.9 |    0.026   |